In [70]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score

df = pd.read_csv('../data/IPL_cleaned.csv', parse_dates = ['date'])


# Match level dataset

In [71]:
matches = df.drop_duplicates(subset = 'match_id')[['match_id','date','season','venue','city','batting_team','bowling_team','toss_winner','toss_decision','match_won_by']].copy()

#Remove no result matches
matches = matches[matches['match_won_by']!='Unknown']

# sort by date
matches = matches.sort_values('date').reset_index(drop=True)


matches['team1']=matches['batting_team']
matches['team2']=matches['bowling_team']

matches.shape

(1146, 12)

In [72]:
def get_h2h_win_pct(row, matches_df):
    prior = matches_df[
        (matches_df['date'] < row['date']) &
        (
            ((matches_df['team1'] == row['team1']) & (matches_df['team2'] == row['team2'])) |
            ((matches_df['team1'] == row['team2']) & (matches_df['team2'] == row['team1']))
        )
    ]
    
    if len(prior) == 0:
        return 0.5
    
    team1_wins = (prior['match_won_by'] == row['team1']).sum()
    return team1_wins / len(prior)

matches['h2h_win_pct_team1'] = matches.apply(lambda row: get_h2h_win_pct(row, matches), axis=1)

print(matches[['date', 'team1', 'team2', 'h2h_win_pct_team1']].tail(10))

           date                        team1                        team2  \
1136 2025-05-23          Sunrisers Hyderabad  Royal Challengers Bengaluru   
1137 2025-05-24                 Punjab Kings               Delhi Capitals   
1138 2025-05-25          Chennai Super Kings               Gujarat Titans   
1139 2025-05-25          Sunrisers Hyderabad        Kolkata Knight Riders   
1140 2025-05-26               Mumbai Indians                 Punjab Kings   
1141 2025-05-27         Lucknow Super Giants  Royal Challengers Bengaluru   
1142 2025-05-29                 Punjab Kings  Royal Challengers Bengaluru   
1143 2025-05-30               Mumbai Indians               Gujarat Titans   
1144 2025-06-01               Mumbai Indians                 Punjab Kings   
1145 2025-06-03  Royal Challengers Bengaluru                 Punjab Kings   

      h2h_win_pct_team1  
1136           0.521739  
1137           0.531250  
1138           0.428571  
1139           0.321429  
1140           0.54838

In [73]:
def get_recent_form(row, matches_df, team_name, n=5):
    # Is team ke pichle N match dhoondo (is match se pehle)
    team_matches = matches_df[
        (matches_df['date'] < row['date']) &
        ((matches_df['team1'] == team_name) | (matches_df['team2'] == team_name))
    ].tail(n)
    
    if len(team_matches) == 0:
        return 0.5
    
    wins = (team_matches['match_won_by'] == team_name).sum()
    return wins / len(team_matches)

matches['team1_recent_form'] = matches.apply(lambda row: get_recent_form(row, matches, row['team1']), axis=1)
matches['team2_recent_form'] = matches.apply(lambda row: get_recent_form(row, matches, row['team2']), axis=1)

print(matches[['date', 'team1', 'team2', 'team1_recent_form', 'team2_recent_form']].tail(10))

           date                        team1                        team2  \
1136 2025-05-23          Sunrisers Hyderabad  Royal Challengers Bengaluru   
1137 2025-05-24                 Punjab Kings               Delhi Capitals   
1138 2025-05-25          Chennai Super Kings               Gujarat Titans   
1139 2025-05-25          Sunrisers Hyderabad        Kolkata Knight Riders   
1140 2025-05-26               Mumbai Indians                 Punjab Kings   
1141 2025-05-27         Lucknow Super Giants  Royal Challengers Bengaluru   
1142 2025-05-29                 Punjab Kings  Royal Challengers Bengaluru   
1143 2025-05-30               Mumbai Indians               Gujarat Titans   
1144 2025-06-01               Mumbai Indians                 Punjab Kings   
1145 2025-06-03  Royal Challengers Bengaluru                 Punjab Kings   

      team1_recent_form  team2_recent_form  
1136                0.4                0.8  
1137                0.8                0.2  
1138             

In [74]:
def get_venue_batting_first_win_pct(row, matches_df):
    prior = matches_df[
        (matches_df['date'] < row['date']) &
        (matches_df['venue'] == row['venue'])
    ]
    
    if len(prior) < 5:  # bahut kam data hai toh neutral rakho
        return 0.5
    
    batting_first_wins = (prior['batting_team'] == prior['match_won_by']).sum()
    return batting_first_wins / len(prior)

matches['venue_batting_first_win_pct'] = matches.apply(lambda row: get_venue_batting_first_win_pct(row, matches), axis=1)

print(matches[['date', 'venue', 'venue_batting_first_win_pct']].tail(10))

           date                                              venue  \
1136 2025-05-23  Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...   
1137 2025-05-24                     Sawai Mansingh Stadium, Jaipur   
1138 2025-05-25                   Narendra Modi Stadium, Ahmedabad   
1139 2025-05-25                        Arun Jaitley Stadium, Delhi   
1140 2025-05-26                     Sawai Mansingh Stadium, Jaipur   
1141 2025-05-27  Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...   
1142 2025-05-29  Maharaja Yadavindra Singh International Cricke...   
1143 2025-05-30  Maharaja Yadavindra Singh International Cricke...   
1144 2025-06-01                   Narendra Modi Stadium, Ahmedabad   
1145 2025-06-03                   Narendra Modi Stadium, Ahmedabad   

      venue_batting_first_win_pct  
1136                     0.421053  
1137                     0.370968  
1138                     0.487805  
1139                     0.483871  
1140                     0.365079  
1141         

In [75]:
# Toss winner kaun sa team hai (team1 ya team2), aur decision kya thi
matches['toss_winner_is_team1'] = (matches['toss_winner'] == matches['team1']).astype(int)

def get_team_overall_win_pct(row, matches_df,team_name):
    prior = matches_df[
        (matches_df['date']< row['date'])&((matches_df['team1'] == team_name)| (matches_df["team2"] == team_name))
    ]

    if len(prior) < 5:
        return 0.5

    wins = (prior["match_won_by"] == team_name).sum()

    return wins / len(prior)

matches['team1_overall_win_pct'] = matches.apply(lambda row: get_team_overall_win_pct(row, matches, row['team1']), axis=1)
matches['team2_overall_win_pct'] = matches.apply(lambda row: get_team_overall_win_pct(row, matches, row['team2']), axis=1)

In [76]:
def calculate_elo_ratings(matches_df, k=32):
    elo = {team: 1500 for team in pd.concat([matches_df['team1'],matches_df["team2"]]).unique()}

    team1_elo_before, team2_elo_before = [], []

    for idx, row in matches_df.iterrows():
        t1,t2 = row['team1'],row['team2']
        team1_elo_before.append(elo[t1])
        team2_elo_before.append(elo[t2])

        expected1 = 1/(1+10 ** ((elo[t2] - elo[t1]) / 400))
        actual1 = 1 if row["match_won_by"] == t1 else 0

        elo[t1] += k * (actual1 - expected1)
        elo[t2] += k*((1-actual1)- (1-expected1))

    matches_df["team1_elo"] = team1_elo_before
    matches_df["team2_elo"] = team2_elo_before
    return matches_df

matches = calculate_elo_ratings(matches)

In [77]:
team_home_city = df.groupby('batting_team')['city'].agg(lambda x: x.value_counts().index[0])
 
matches['team1_home_advantage'] = matches.apply(
    lambda row: 1 if team_home_city.get(row['team1']) == row['city'] else 0, axis=1
)
matches['team2_home_advantage'] = matches.apply(
    lambda row: 1 if team_home_city.get(row['team2']) == row['city'] else 0, axis=1
)

In [78]:
matches['team1_won'] = (matches['match_won_by'] == matches['team1']).astype(int)
 
print("Feature engineering done. Final columns:", matches.columns.tolist())
 

Feature engineering done. Final columns: ['match_id', 'date', 'season', 'venue', 'city', 'batting_team', 'bowling_team', 'toss_winner', 'toss_decision', 'match_won_by', 'team1', 'team2', 'h2h_win_pct_team1', 'team1_recent_form', 'team2_recent_form', 'venue_batting_first_win_pct', 'toss_winner_is_team1', 'team1_overall_win_pct', 'team2_overall_win_pct', 'team1_elo', 'team2_elo', 'team1_home_advantage', 'team2_home_advantage', 'team1_won']


In [79]:
feature_cols = [
    'team1_elo', 'team2_elo',
    'team1_home_advantage', 'team2_home_advantage',
    'toss_winner_is_team1',
    'h2h_win_pct_team1', 'team1_recent_form', 'team2_recent_form',
    'venue_batting_first_win_pct'
]
 
X = matches[feature_cols]
y = matches['team1_won']


In [80]:
train_data = matches[matches['season'] <= 2023]
test_data = matches[matches['season'] > 2023]
 
X_train, y_train = train_data[feature_cols], train_data['team1_won']
X_test, y_test = test_data[feature_cols], test_data['team1_won']
 
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
 


Train: (1005, 9), Test: (141, 9)


In [81]:
# Baseline check (always predict the majority class)
baseline_accuracy = max(y_test.mean(), 1 - y_test.mean())
print(f"\nBaseline accuracy: {baseline_accuracy:.2%}")
 


Baseline accuracy: 51.77%


In [82]:
# --- XGBoost (regularized to avoid overfitting) ---
xgb_model = XGBClassifier(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    reg_alpha=1, reg_lambda=2, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_model.fit(X_train, y_train)
print(f"\nXGBoost Train Acc: {accuracy_score(y_train, xgb_model.predict(X_train)):.2%}")
print(f"XGBoost Test Acc:  {accuracy_score(y_test, xgb_model.predict(X_test)):.2%}")
 
xgb_cv = cross_val_score(xgb_model, X, y, cv=5)
print(f"XGBoost CV Average: {xgb_cv.mean():.2%} (+/- {xgb_cv.std():.2%})")
 



XGBoost Train Acc: 60.20%
XGBoost Test Acc:  46.81%
XGBoost CV Average: 52.88% (+/- 2.47%)


In [83]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
print(f"\nLogReg Train Acc: {accuracy_score(y_train, lr_model.predict(X_train_scaled)):.2%}")
print(f"LogReg Test Acc:  {accuracy_score(y_test, lr_model.predict(X_test_scaled)):.2%}")
 


LogReg Train Acc: 57.51%
LogReg Test Acc:  48.94%


In [84]:
# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=10, random_state=42)
rf_model.fit(X_train, y_train)
print(f"\nRandom Forest Train Acc: {accuracy_score(y_train, rf_model.predict(X_train)):.2%}")
print(f"Random Forest Test Acc:  {accuracy_score(y_test, rf_model.predict(X_test)):.2%}")
 
rf_cv = cross_val_score(rf_model, X, y, cv=5)
print(f"Random Forest CV Average: {rf_cv.mean():.2%}")
 


Random Forest Train Acc: 62.99%
Random Forest Test Acc:  49.65%
Random Forest CV Average: 52.44%


In [85]:
# ----------------------------------------------------------
# 5. FINDING (for project report)
# ----------------------------------------------------------
# All three models (XGBoost, Logistic Regression, Random Forest) converge to
# ~52-53% cross-validated accuracy, only marginally above the ~52% baseline.
# This is consistent with sports-analytics literature: pre-match T20 outcome
# prediction is inherently limited without live match-state data (playing XI,
# pitch/weather conditions, in-game momentum). The finding itself is valid
# and worth reporting, rather than a failure to be hidden.
# ----------------------------------------------------------
 
# Feature importance (from XGBoost) — useful for the report
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature importance:\n", importance)
 


Feature importance:
                        feature  importance
1                    team2_elo    0.128525
2         team1_home_advantage    0.128002
5            h2h_win_pct_team1    0.127494
6            team1_recent_form    0.125649
0                    team1_elo    0.122579
8  venue_batting_first_win_pct    0.103513
3         team2_home_advantage    0.102500
7            team2_recent_form    0.097824
4         toss_winner_is_team1    0.063914


In [ ]:
matches.to_csv('../data/matches_with_features.csv', index=False)
print("\nSaved matches_with_features.csv")


Saved matches_with_features.csv


: 